# Logistics Operations — Data Exploration
**Project:** Logistics Capstone  
**Notebook:** 01 — Load & Explore All Sheets  
**Tools:** pandas  

---
**Goal:** Load all 14 sheets, understand their shape, spot any quality issues, and map the relationships before we start merging or analysing anything.

## 1. Import libraries

In [ ]:
import pandas as pd

# Show all columns when printing a DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('pandas version:', pd.__version__)

## 2. Load all 14 sheets at once

`sheet_name=None` tells pandas to load every sheet and return them as a **dictionary**  
where each key is the sheet name and each value is a DataFrame.

Update `FILE_PATH` to match where you saved the Excel file on your Mac.

In [ ]:
FILE_PATH = '../data/raw/Logistics Operations Database 2.xlsx'  # update if needed

# Load every sheet into a dict:  {'SHEET_NAME': DataFrame, ...}
all_sheets = pd.read_excel(FILE_PATH, sheet_name=None)

print(f'Sheets loaded: {len(all_sheets)}')
print('Sheet names:')
for name in all_sheets:
    print(f'  - {name}')

## 3. Assign each sheet to a named variable

Working with `all_sheets['TRIPS']` everywhere gets messy.  
We assign each sheet to a short, clean variable name.

> **Note:** Update the key names below to match the exact sheet names printed above.

In [ ]:
# Assign each sheet — key names must match exactly what printed above
drivers             = all_sheets['DRIVERS']
trucks              = all_sheets['TRUCKS']
trailers            = all_sheets['TRAILERS']
customers           = all_sheets['CUSTOMERS']
facilities          = all_sheets['FACILITIES']
routes              = all_sheets['ROUTES']
loads               = all_sheets['LOADS']
trips               = all_sheets['TRIPS']
fuel_purchases      = all_sheets['FUEL_PURCHASES']
maintenance         = all_sheets['MAINTENANCE_RECORDS']
delivery_events     = all_sheets['DELIVERY_EVENTS']
safety_incidents    = all_sheets['SAFETY_INCIDENTS']
driver_metrics      = all_sheets['DRIVER_MONTHLY_METRICS']
truck_metrics       = all_sheets['TRUCK_UTILIZATION_METRICS']

print('All sheets assigned successfully.')

## 4. Quick overview of every sheet

This loop gives us a one-line summary for each table:  
how many rows, how many columns, and how many null values exist.

In [ ]:
print(f'{"Table":<35} {"Rows":>8} {"Cols":>6} {"Nulls":>8}')
print('-' * 62)

for name, df in all_sheets.items():
    nulls = df.isnull().sum().sum()
    print(f'{name:<35} {df.shape[0]:>8,} {df.shape[1]:>6} {nulls:>8,}')

## 5. Deep dive — explore each table

For every table we look at:
- `.head()` — first 5 rows to see what data looks like
- `.info()` — column names, data types, non-null counts
- `.describe()` — stats for numeric columns

Run each cell and **add notes** about anything that looks odd.

### 5a. DRIVERS

In [ ]:
print('Shape:', drivers.shape)
drivers.head()

In [ ]:
drivers.info()

In [ ]:
drivers.describe()

### 5b. TRUCKS

In [ ]:
print('Shape:', trucks.shape)
trucks.head()

In [ ]:
trucks.info()

### 5c. TRAILERS

In [ ]:
print('Shape:', trailers.shape)
trailers.head()

### 5d. CUSTOMERS

In [ ]:
print('Shape:', customers.shape)
customers.head()

In [ ]:
customers.describe()

### 5e. FACILITIES

In [ ]:
print('Shape:', facilities.shape)
facilities.head()

### 5f. ROUTES

In [ ]:
print('Shape:', routes.shape)
routes.head()

In [ ]:
routes.describe()

### 5g. LOADS  *(has foreign keys: customer_id, route_id)*

In [ ]:
print('Shape:', loads.shape)
loads.head()

In [ ]:
loads.info()

In [ ]:
loads.describe()

### 5h. TRIPS  *(central table — joins to almost everything)*

In [ ]:
print('Shape:', trips.shape)
trips.head()

In [ ]:
trips.info()

In [ ]:
trips.describe()

### 5i. FUEL_PURCHASES

In [ ]:
print('Shape:', fuel_purchases.shape)
fuel_purchases.head()

In [ ]:
fuel_purchases.describe()

### 5j. MAINTENANCE_RECORDS

In [ ]:
print('Shape:', maintenance.shape)
maintenance.head()

In [ ]:
maintenance.describe()

### 5k. DELIVERY_EVENTS

In [ ]:
print('Shape:', delivery_events.shape)
delivery_events.head()

In [ ]:
delivery_events.info()

### 5l. SAFETY_INCIDENTS

In [ ]:
print('Shape:', safety_incidents.shape)
safety_incidents.head()

In [ ]:
safety_incidents.describe()

### 5m. DRIVER_MONTHLY_METRICS  *(pre-aggregated — use for trends)*

In [ ]:
print('Shape:', driver_metrics.shape)
driver_metrics.head()

In [ ]:
driver_metrics.describe()

### 5n. TRUCK_UTILIZATION_METRICS  *(pre-aggregated — use for fleet trends)*

In [ ]:
print('Shape:', truck_metrics.shape)
truck_metrics.head()

In [ ]:
truck_metrics.describe()

## 6. Null value breakdown — per table

We already saw the *total* nulls per table above.  
This cell shows which *columns* have nulls so we know what to fix in the cleaning notebook.

In [ ]:
for name, df in all_sheets.items():
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0]  # only show columns that have nulls
    if len(null_cols) > 0:
        print(f'\n--- {name} ---')
        print(null_cols.to_string())

print('\nDone. Any table not listed above has zero nulls.')

## 7. Verify key relationships

Before merging, we check that foreign keys actually match between tables.  
If the overlap is 100%, the join will be clean. If not, we have orphan records to investigate.

Think of this as the pandas version of a SQL referential integrity check.

In [ ]:
def check_fk(child_df, child_col, parent_df, parent_col, label):
    """Check how many FK values in child_df exist in parent_df."""
    child_vals  = set(child_df[child_col].dropna())
    parent_vals = set(parent_df[parent_col].dropna())
    matched     = child_vals & parent_vals
    pct         = len(matched) / len(child_vals) * 100 if child_vals else 0
    status      = 'OK' if pct == 100 else 'WARN'
    print(f'[{status}] {label}: {pct:.1f}% of {len(child_vals):,} values matched')

check_fk(loads,           'customer_id', customers,  'customer_id', 'loads -> customers')
check_fk(loads,           'route_id',    routes,     'route_id',    'loads -> routes')
check_fk(trips,           'load_id',     loads,      'load_id',     'trips -> loads')
check_fk(trips,           'driver_id',   drivers,    'driver_id',   'trips -> drivers')
check_fk(trips,           'truck_id',    trucks,     'truck_id',    'trips -> trucks')
check_fk(trips,           'trailer_id',  trailers,   'trailer_id',  'trips -> trailers')
check_fk(fuel_purchases,  'trip_id',     trips,      'trip_id',     'fuel_purchases -> trips')
check_fk(maintenance,     'truck_id',    trucks,     'truck_id',    'maintenance -> trucks')
check_fk(delivery_events, 'trip_id',     trips,      'trip_id',     'delivery_events -> trips')
check_fk(delivery_events, 'facility_id', facilities, 'facility_id', 'delivery_events -> facilities')
check_fk(safety_incidents,'trip_id',     trips,      'trip_id',     'safety_incidents -> trips')

## 8. Notes & observations

Use this cell to record anything you noticed while running the above.  
This becomes your data quality log before cleaning.

**Template — fill in after running all cells above:**

```
DRIVERS      — rows: ?, cols: ?, nulls: ?
TRUCKS       — rows: ?, cols: ?, nulls: ?
TRAILERS     — rows: ?, cols: ?, nulls: ?
CUSTOMERS    — rows: ?, cols: ?, nulls: ?
FACILITIES   — rows: ?, cols: ?, nulls: ?
ROUTES       — rows: ?, cols: ?, nulls: ?
LOADS        — rows: ?, cols: ?, nulls: ?
TRIPS        — rows: ?, cols: ?, nulls: ?
FUEL_PUR.    — rows: ?, cols: ?, nulls: ?
MAINTENANCE  — rows: ?, cols: ?, nulls: ?
DELIVERY_EV. — rows: ?, cols: ?, nulls: ?
SAFETY_INC.  — rows: ?, cols: ?, nulls: ?
DRV_METRICS  — rows: ?, cols: ?, nulls: ?
TRK_METRICS  — rows: ?, cols: ?, nulls: ?

Issues to fix in 02_cleaning.ipynb:
- 
- 
-
```

---
**Next notebook:** `02_cleaning.ipynb` — fix nulls, data types, and prepare merge-ready tables.